# Assignment 4 - Fraud Detection (Prediction Notebook)
### Using scikit-learn MLPClassifier — no TensorFlow required

**Instructions:**
1. Set `TEST_FILE` below to your test CSV path.
2. Ensure `fraud_model.pkl`, `scaler.pkl`, and `feature_names.pkl` are in the same folder.
3. Click **Run All**.

In [ ]:
# ── USER CONFIGURATION ──────────────────────────────────────────────
TEST_FILE = "fraudTest.csv.zip"   # <-- change to your test file path
# ────────────────────────────────────────────────────────────────────

In [ ]:
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from geopy import distance
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

print('All imports OK')

## Step 1 — Load Saved Model & Preprocessing Artifacts

In [ ]:
model         = joblib.load('fraud_model.pkl')
scaler        = joblib.load('scaler.pkl')
feature_names = joblib.load('feature_names.pkl')

print("Model loaded:", type(model))
print(f"Expecting {len(feature_names)} features")

## Step 2 — Data Preparation Function
Applies the same transformations used during training.

In [ ]:
def compute_distance(row):
    home  = (row['lat'],       row['long'])
    merch = (row['merch_lat'], row['merch_long'])
    return distance.distance(home, merch).km


def prepare_data(df_raw):
    """
    Feature-engineer a raw DataFrame with the same schema as fraudTrain.csv.
    Returns X_scaled (numpy array) and y (array of true labels, or None).
    """
    df = df_raw.copy()
    df.drop(columns={'Unnamed: 0'}, inplace=True, errors='ignore')

    # Location
    print("Computing distances...")
    df['dist_km'] = df.apply(compute_distance, axis=1)

    # Time
    df['trans_dt'] = pd.to_datetime(df['trans_date_trans_time'])
    df['dob_dt']   = pd.to_datetime(df['dob'])
    hour = df['trans_dt'].dt.hour
    dow  = df['trans_dt'].dt.dayofweek
    df['hour_sin'] = np.sin(2 * np.pi * hour / 24)
    df['hour_cos'] = np.cos(2 * np.pi * hour / 24)
    df['dow_sin']  = np.sin(2 * np.pi * dow  / 7)
    df['dow_cos']  = np.cos(2 * np.pi * dow  / 7)
    df['age']      = (df['trans_dt'] - df['dob_dt']).dt.days / 365.25

    # Drop raw columns
    drop_cols = [
        'trans_date_trans_time', 'trans_dt', 'dob', 'dob_dt',
        'cc_num', 'merchant', 'first', 'last', 'street',
        'city', 'state', 'zip', 'job', 'trans_num',
        'lat', 'long', 'merch_lat', 'merch_long', 'unix_time'
    ]
    df.drop(columns=drop_cols, inplace=True, errors='ignore')

    # Separate target if present
    if 'is_fraud' in df.columns:
        y = df.pop('is_fraud').values.astype(np.float32)
    else:
        y = None

    # One-hot encode
    df = pd.get_dummies(df, columns=['category', 'gender'], drop_first=False)

    # Align columns with training set
    for col in feature_names:
        if col not in df.columns:
            df[col] = 0
    df = df[feature_names]

    X_scaled = scaler.transform(df.values.astype(np.float32))
    return X_scaled, y

## Step 3 — Load & Prepare Test Data

In [ ]:
print(f"Loading: {TEST_FILE}")
df_test = pd.read_csv(TEST_FILE)
print(f"Rows: {len(df_test):,}   Columns: {df_test.shape[1]}")

X_test, y_test = prepare_data(df_test)
print(f"Feature matrix: {X_test.shape}")

## Step 4 — Make Predictions

In [ ]:
print("Running predictions...")
y_pred      = model.predict(X_test)
y_pred_prob = model.predict_proba(X_test)[:, 1]

print(f"Predicted fraud : {y_pred.sum():,}")
print(f"Predicted legit : {(y_pred == 0).sum():,}")

## Step 5 — Confusion Matrix & Classification Report

In [ ]:
if y_test is not None:
    cm = confusion_matrix(y_test, y_pred)

    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Legit', 'Fraud'])
    fig, ax = plt.subplots(figsize=(6, 5))
    disp.plot(cmap='Blues', ax=ax)
    ax.set_title('Confusion Matrix — Test Set')
    plt.tight_layout()
    plt.show()

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=['Legit', 'Fraud']))

    tn, fp, fn, tp = cm.ravel()
    print(f"True Negatives  (Legit correctly identified) : {tn:>8,}")
    print(f"False Positives (Legit flagged as Fraud)     : {fp:>8,}")
    print(f"False Negatives (Fraud missed)               : {fn:>8,}")
    print(f"True Positives  (Fraud correctly caught)     : {tp:>8,}")
else:
    print("No 'is_fraud' column in test file — skipping confusion matrix.")
    print("Predictions are in y_pred (0=Legit, 1=Fraud).")

## Step 6 — Export Predictions (Optional)

In [ ]:
output = pd.DataFrame({
    'predicted_label':   y_pred,
    'fraud_probability': y_pred_prob
})
if y_test is not None:
    output['true_label'] = y_test.astype(int)

output.to_csv('predictions.csv', index=False)
print("Predictions saved to predictions.csv")
output.head(10)